# Which Hangboard Protocol Suits Me?

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/8cH9azbsFifZ/climbing-science/main?labpath=notebooks%2F04_protocol_comparison.ipynb)


## Introduction

Hangboard training is the most researched method for improving
finger strength in climbing. Yet the number of published protocols
can be overwhelming — MaxHangs, Repeaters, Density Hangs, and many
more.

This notebook helps you **compare all protocols** in the
`climbing_science` registry side by side:

- **Time Under Tension (TUT)** per set and per session
- **Energy system** targeted (alactic / lactic / aerobic)
- **Minimum level** required
- **Personalised recommendations** based on your weakness profile

Key references:
López-Rivera & González-Badillo (2012), Hörst (2016), Nelson, Bechtel, Anderson.


## Your Data

**Edit the cell below** — this is the only cell you need to change.


In [ ]:
# ── YOUR PARAMETERS (edit here) ──────────────────────────────
MVC7_KG        = 105          # 7-second max voluntary contraction [kg]
BODYWEIGHT_KG  = 72           # body weight [kg]
CLIMBING_LEVEL = "intermediate"  # beginner | intermediate | advanced | elite


In [ ]:
%matplotlib inline

from climbing_science.protocols import (
    get_protocol,
    list_protocols,
    select_protocols,
    format_notation,
)
from climbing_science.load import tut_per_set, tut_per_session
from climbing_science.frontends.notebook import plot_protocol_comparison
from climbing_science.models import ClimberLevel, EnergySystem

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from IPython.display import display


## All Protocols at a Glance

The protocol registry ships with every published hangboard protocol
we could find. Each row shows the key training parameters.


In [ ]:
protocols = list_protocols()
print(f"{len(protocols)} protocols in registry\n")

rows = []
for p in protocols:
    rows.append({
        "ID": p.id,
        "Name": p.name,
        "Author": p.author,
        "Hang [s]": p.params.hang_duration_sec,
        "Reps": p.params.reps_per_set,
        "Sets": p.params.sets,
        "Rest reps [s]": p.params.rest_between_reps_sec,
        "Rest sets [s]": p.params.rest_between_sets_sec,
        "Intensity [%MVC]": f"{p.params.intensity_percent_mvc[0]:.0f}–{p.params.intensity_percent_mvc[1]:.0f}",
        "Energy": p.energy_system.value,
        "Min Level": p.min_level.value,
    })

if HAS_PANDAS:
    df_all = pd.DataFrame(rows)
    display(df_all)
else:
    # Plain-text fallback
    header = list(rows[0].keys())
    print("  |  ".join(header))
    print("-" * 120)
    for r in rows:
        print("  |  ".join(str(r[k]) for k in header))


## Filter by Your Level

Not every protocol is appropriate for every climber.
We filter protocols whose minimum level is at or below yours.


In [ ]:
level_order = [l.value for l in ClimberLevel]
my_idx = level_order.index(CLIMBING_LEVEL)

suitable = [
    p for p in protocols
    if level_order.index(p.min_level.value) <= my_idx
]
print(f"{len(suitable)} protocols suitable for level '{CLIMBING_LEVEL}':\n")

if HAS_PANDAS:
    rows_s = [{
        "Name": p.name,
        "Author": p.author,
        "Energy": p.energy_system.value,
        "Hang [s]": p.params.hang_duration_sec,
        "Sets × Reps": f"{p.params.sets}×{p.params.reps_per_set}",
        "Min Level": p.min_level.value,
    } for p in suitable]
    display(pd.DataFrame(rows_s))
else:
    for p in suitable:
        print(f"  • {p.name} ({p.author}) — {p.energy_system.value}")


## Protocol Comparison Plot

The built-in comparison plot shows how each protocol maps onto
absolute load (from your MVC-7) and time under tension.


In [ ]:
plot_protocol_comparison(MVC7_KG, BODYWEIGHT_KG)


## Time Under Tension (TUT) Comparison

TUT is the total seconds your fingers are loaded. It is a simple
proxy for training stimulus volume.

- **TUT per set** = hang duration × reps per set
- **TUT per session** = TUT per set × number of sets


In [ ]:
names, tuts_set, tuts_session = [], [], []

for p in protocols:
    names.append(p.name)
    tuts_set.append(
        tut_per_set(p.params.hang_duration_sec, p.params.reps_per_set,
                    p.params.rest_between_reps_sec)
    )
    tuts_session.append(
        tut_per_session(p.params.hang_duration_sec, p.params.reps_per_set,
                        p.params.sets)
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TUT per set
axes[0].barh(names, tuts_set, color="steelblue")
axes[0].set_xlabel("TUT per set [s]")
axes[0].set_title("TUT per Set")
axes[0].invert_yaxis()

# TUT per session
axes[1].barh(names, tuts_session, color="coral")
axes[1].set_xlabel("TUT per session [s]")
axes[1].set_title("TUT per Session")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## Energy System Classification

Hangboard protocols target different energy systems:

| Energy System | Duration | Mechanism | Example |
|---|---|---|---|
| **Alactic** (ATP-CP) | < 15 s | Phosphocreatine | MaxHangs |
| **Lactic** (glycolytic) | 15–120 s | Anaerobic glycolysis | Repeaters, IntHangs |
| **Aerobic** (oxidative) | > 120 s | Oxidative phosphorylation | Density Hangs, ARC |

Training the right system depends on your weakness:
- **Weak fingers?** → Alactic protocols build peak strength
- **Pump out early?** → Lactic protocols improve power-endurance
- **Can't recover between burns?** → Aerobic protocols boost capillarisation


In [ ]:
# Group protocols by energy system
by_energy = {}
for p in protocols:
    key = p.energy_system.value
    by_energy.setdefault(key, []).append(p)

for system in ["alactic", "lactic", "aerobic"]:
    group = by_energy.get(system, [])
    print(f"\n{'='*50}")
    print(f"  {system.upper()} — {len(group)} protocol(s)")
    print(f"{'='*50}")
    for p in group:
        tut_s = tut_per_session(
            p.params.hang_duration_sec,
            p.params.reps_per_set,
            p.params.sets,
        )
        print(f"  • {p.name:35s}  TUT/session={tut_s:6.0f}s  "
              f"level≥{p.min_level.value}")


## Decision Tree — Which Protocol for Which Weakness?

```
What is your primary weakness?
│
├─ Weak fingers (can't hold small edges)
│  └─ MaxHangs: López MAW, López HAW, Hörst 7-53
│     → alactic, high intensity, low volume
│
├─ Poor power-endurance (pump out on crux sequences)
│  └─ Repeaters: López IntHangs, Bechtel, Nelson
│     → lactic, moderate intensity, high volume
│
├─ Poor endurance (can't sustain on long routes)
│  └─ Density Hangs, ARC-style protocols
│     → aerobic, low intensity, very high volume
│
└─ Plateau (no progress despite training)
   └─ Switch energy system!
      If you've been doing MaxHangs → try Repeaters
      If you've been doing Repeaters → try MaxHangs
```

The library encodes this logic in `select_protocols()`:


In [ ]:
level = ClimberLevel(CLIMBING_LEVEL)

for weakness in ["strength", "endurance", "power-endurance", "strength-endurance"]:
    recs = select_protocols(weakness, level)
    print(f"\nWeakness: {weakness!r} → {len(recs)} recommendation(s)")
    for p in recs:
        print(f"  ✓ {p.name} ({p.energy_system.value})")


## Protocol Notation

`format_notation()` produces a compact, human-readable description
of each protocol. This is the shorthand you can write on your
training log.


In [ ]:
for p in protocols:
    notation = format_notation(p)
    print(f"{p.name:40s}  {notation}")


## References

- **López-Rivera, E. & González-Badillo, J. J.** (2012). *The effects of two maximum grip strength training methods using the same effort duration and rest time on grip endurance in elite climbers.* Sports Technology, 5(3–4), 98–110.
- **Hörst, E. J.** (2016). *Training for Climbing* (3rd ed.). Falcon Guides.
- **Nelson, M.** Hangboard training protocols for climbing. Tension Climbing.
- **Bechtel, S.** Hangboard ladders and repeaters. Climb Strong.
- **Anderson, M. & Anderson, M.** *The Rock Climber's Training Manual.* Fixed Pin Publishing.
- **López-Rivera, E.** (2014). *Efectos del entrenamiento de fuerza…* Doctoral dissertation.
